# Notebook 5: Mock Interview — Timed Practice Challenges

**This notebook simulates the actual interview.** Each challenge gives you:
- A scenario description
- Starter code (like you'll get in Colab)
- Specific requirements
- A time target
- A solution you can check after

**Rules:**
- Use Google/docs/Stack Overflow (open book)
- NO LLM coding assistants
- Write ALL the code yourself
- Set a timer

---

## Starter Code (Provided in Interview)

This is the kind of starter code you'll get. Read it carefully — the patterns you need are here.

In [ ]:
# ============================================
# STARTER CODE — This is provided for you
# ============================================
!pip install anthropic -q

import anthropic
import json

# Initialize the client
# In Colab, you'll set your API key here:
# client = anthropic.Anthropic(api_key="sk-ant-...")
client = anthropic.Anthropic()

# ---- TOOL DEFINITION TEMPLATE ----
# Tools are defined as a list of dictionaries with:
#   - name: tool identifier
#   - description: what the tool does (be detailed!)
#   - input_schema: JSON Schema for parameters

example_tool = {
    "name": "get_weather",
    "description": "Get the current weather in a given location",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city and state, e.g. San Francisco, CA"
            }
        },
        "required": ["location"]
    }
}

# ---- BASIC API CALL ----
# response = client.messages.create(
#     model="claude-sonnet-4-20250514",
#     max_tokens=1024,
#     tools=[example_tool],
#     messages=[{"role": "user", "content": "What's the weather in SF?"}]
# )

# ---- RESPONSE STRUCTURE ----
# response.stop_reason: "end_turn" (done) or "tool_use" (wants to use a tool)
# response.content: list of content blocks (text or tool_use)
# tool_use block: {type: "tool_use", id: "toolu_xxx", name: "tool_name", input: {...}}

# ---- TOOL RESULT FORMAT ----
# {"type": "tool_result", "tool_use_id": "toolu_xxx", "content": "result string"}

# ---- AGENTIC LOOP SKELETON ----
# messages = [{"role": "user", "content": "..."}]
# while True:
#     response = client.messages.create(...)
#     if response.stop_reason == "end_turn": break
#     # process tool calls, add results, continue

print("Starter code loaded. Ready for challenges.")

---

## Challenge 1: Customer Support Agent (Target: 25 min)

### Scenario
Build a customer support agent for an e-commerce company. The agent should help customers check orders, find products, and handle returns.

### Requirements
1. Define **4 tools**:
   - `lookup_order(order_id)` — Returns order status, items, and total
   - `search_products(query, category?)` — Search product catalog
   - `initiate_return(order_id, reason)` — Start a return process
   - `get_customer_info(customer_id)` — Get customer profile

2. Implement the **handler functions** (use fake data)

3. Build the **agentic loop** that handles multi-step conversations

4. Test with these queries:
   - "Can you check on my order ORD-123?"
   - "I want to return order ORD-123 because it arrived damaged"
   - "Do you have any wireless headphones?"

### Start your timer now. GO!

In [ ]:
# ============================================
# CHALLENGE 1: YOUR CODE HERE
# ============================================

# 1. Tool handler functions (fake implementations)


# 2. Tool definitions (JSON schemas)


# 3. Dispatcher


# 4. Agentic loop


# 5. Test


In [ ]:
# ============================================
# CHALLENGE 1: SOLUTION
# ============================================

# 1. Handler functions
def lookup_order(order_id):
    orders = {
        "ORD-123": {
            "order_id": "ORD-123",
            "status": "delivered",
            "items": [{"name": "Wireless Mouse", "qty": 1, "price": 29.99},
                      {"name": "USB-C Cable", "qty": 2, "price": 12.99}],
            "total": 55.97,
            "delivered_date": "2025-01-10"
        },
        "ORD-456": {
            "order_id": "ORD-456",
            "status": "in_transit",
            "items": [{"name": "Keyboard", "qty": 1, "price": 79.99}],
            "total": 79.99,
            "tracking": "1Z999AA10123456784"
        }
    }
    return json.dumps(orders.get(order_id, {"error": f"Order {order_id} not found"}))

def search_products(query, category=None):
    products = [
        {"id": "P-001", "name": "Wireless Bluetooth Headphones", "price": 49.99, "category": "audio"},
        {"id": "P-002", "name": "Noise Cancelling Earbuds", "price": 89.99, "category": "audio"},
        {"id": "P-003", "name": "Wireless Mouse", "price": 29.99, "category": "peripherals"},
        {"id": "P-004", "name": "Mechanical Keyboard", "price": 79.99, "category": "peripherals"},
    ]
    results = [p for p in products if query.lower() in p["name"].lower()]
    if category:
        results = [p for p in results if p["category"] == category]
    return json.dumps({"results": results, "count": len(results)})

def initiate_return(order_id, reason):
    return json.dumps({
        "return_id": "RET-" + order_id.split("-")[1],
        "order_id": order_id,
        "reason": reason,
        "status": "initiated",
        "instructions": "Please ship the item back within 14 days. A prepaid label will be emailed."
    })

def get_customer_info(customer_id):
    customers = {
        "CUST-001": {"name": "Alice Johnson", "email": "alice@example.com", "tier": "gold"},
    }
    return json.dumps(customers.get(customer_id, {"error": "Customer not found"}))


# 2. Tool definitions
cs_tools = [
    {
        "name": "lookup_order",
        "description": (
            "Look up an order by its ID. Returns order status, line items with prices, "
            "total amount, and delivery/tracking info. Use when a customer asks about an order."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID, e.g. ORD-123"}
            },
            "required": ["order_id"]
        }
    },
    {
        "name": "search_products",
        "description": (
            "Search the product catalog by keyword. Optionally filter by category. "
            "Returns matching products with IDs, names, prices. Use when a customer asks about products."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search keyword"},
                "category": {"type": "string", "description": "Optional category filter"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "initiate_return",
        "description": (
            "Start a return process for an order. Creates a return ticket and provides "
            "instructions. Use when a customer wants to return an order."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID to return"},
                "reason": {"type": "string", "description": "Reason for the return"}
            },
            "required": ["order_id", "reason"]
        }
    },
    {
        "name": "get_customer_info",
        "description": "Get customer profile by ID. Returns name, email, membership tier.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "Customer ID, e.g. CUST-001"}
            },
            "required": ["customer_id"]
        }
    }
]


# 3. Dispatcher
HANDLERS = {
    "lookup_order": lookup_order,
    "search_products": search_products,
    "initiate_return": initiate_return,
    "get_customer_info": get_customer_info,
}

def process_tool(name, input_data):
    handler = HANDLERS.get(name)
    if not handler:
        return f"Unknown tool: {name}"
    try:
        return handler(**input_data)
    except Exception as e:
        return f"Error: {e}"


# 4. Agentic loop
def run_support_agent(user_message, max_turns=10):
    messages = [{"role": "user", "content": user_message}]
    
    for _ in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            tools=cs_tools,
            system="You are a helpful customer support agent. Be concise and professional.",
            messages=messages
        )
        
        if response.stop_reason == "end_turn":
            return next((b.text for b in response.content if b.type == "text"), "")
        
        messages.append({"role": "assistant", "content": response.content})
        
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = process_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result)
                })
        
        messages.append({"role": "user", "content": tool_results})
    
    return "Max turns reached"


# 5. Test
print("=== Test 1: Order lookup ===")
print(run_support_agent("Can you check on my order ORD-123?"))

print("\n=== Test 2: Return ===")
print(run_support_agent("I want to return order ORD-123 because it arrived damaged"))

print("\n=== Test 3: Product search ===")
print(run_support_agent("Do you have any wireless headphones?"))

---

## Challenge 2: Data Analysis Agent (Target: 30 min)

### Scenario
Build a data analysis agent that can query a database, run calculations, and generate insights.

### Requirements
1. Define **5 tools**:
   - `query_database(sql)` — Execute a SQL query (fake, returns canned data)
   - `calculate(expression)` — Evaluate a math expression
   - `create_chart(data, chart_type, title)` — Generate chart (fake, returns description)
   - `get_table_schema(table_name)` — Get column info for a table
   - `save_report(title, content)` — Save findings as a report

2. Build the agent with **state** (reports list, query history)

3. Add **validation**: SQL queries should be read-only (no DROP, DELETE, etc.)

4. Test with:
   - "What tables are in the database? Show me the schema for the sales table."
   - "How many sales were made in Q4 2024? Create a chart of monthly sales."

### Start your timer. GO!

In [ ]:
# ============================================
# CHALLENGE 2: YOUR CODE HERE
# ============================================



In [ ]:
# ============================================
# CHALLENGE 2: SOLUTION
# ============================================

class DataAnalysisAgent:
    def __init__(self):
        self.reports = []
        self.query_history = []
        
        self.tools = [
            {
                "name": "query_database",
                "description": (
                    "Execute a read-only SQL query against the database. Returns results as rows. "
                    "Only SELECT queries are allowed. Use get_table_schema first to understand the data."
                ),
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "sql": {"type": "string", "description": "The SQL SELECT query to execute"}
                    },
                    "required": ["sql"]
                }
            },
            {
                "name": "calculate",
                "description": "Evaluate a mathematical expression. Supports +, -, *, /, **, and common functions.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "expression": {"type": "string", "description": "Math expression, e.g. '(100 * 0.15) + 50'"}
                    },
                    "required": ["expression"]
                }
            },
            {
                "name": "create_chart",
                "description": "Generate a chart visualization from data. Returns a description of the chart created.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "data": {"type": "string", "description": "JSON string of data to chart"},
                        "chart_type": {"type": "string", "enum": ["bar", "line", "pie"], "description": "Type of chart"},
                        "title": {"type": "string", "description": "Chart title"}
                    },
                    "required": ["data", "chart_type", "title"]
                }
            },
            {
                "name": "get_table_schema",
                "description": "Get the column names, types, and descriptions for a database table.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "table_name": {"type": "string", "description": "Name of the table"}
                    },
                    "required": ["table_name"]
                }
            },
            {
                "name": "save_report",
                "description": "Save analysis findings as a report. Returns a confirmation with report ID.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "title": {"type": "string", "description": "Report title"},
                        "content": {"type": "string", "description": "Report content/findings"}
                    },
                    "required": ["title", "content"]
                }
            }
        ]
    
    def query_database(self, sql):
        # Validation: read-only
        dangerous = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "TRUNCATE", "CREATE"]
        if any(d in sql.upper() for d in dangerous):
            return json.dumps({"error": "Only SELECT queries are allowed"})
        
        self.query_history.append(sql)
        
        # Canned responses based on query keywords
        if "COUNT" in sql.upper() and "sales" in sql.lower():
            return json.dumps({"columns": ["count"], "rows": [[1247]], "query": sql})
        if "monthly" in sql.lower() or "GROUP BY" in sql.upper():
            return json.dumps({
                "columns": ["month", "total_sales", "num_orders"],
                "rows": [
                    ["Oct 2024", 125000, 312],
                    ["Nov 2024", 189000, 478],
                    ["Dec 2024", 245000, 612],
                ]
            })
        return json.dumps({"columns": ["id", "date", "amount"], "rows": [[1, "2024-12-01", 99.99]], "query": sql})
    
    def calculate(self, expression):
        try:
            # Safe eval for math only
            allowed = set('0123456789+-*/.() ')
            if not all(c in allowed for c in expression):
                return json.dumps({"error": "Expression contains invalid characters"})
            result = eval(expression)
            return json.dumps({"expression": expression, "result": result})
        except Exception as e:
            return json.dumps({"error": str(e)})
    
    def create_chart(self, data, chart_type, title):
        return json.dumps({
            "chart_id": "CHART-001",
            "type": chart_type,
            "title": title,
            "status": "created",
            "description": f"A {chart_type} chart titled '{title}' has been generated with the provided data."
        })
    
    def get_table_schema(self, table_name):
        schemas = {
            "sales": {
                "table": "sales",
                "columns": [
                    {"name": "id", "type": "INTEGER", "description": "Primary key"},
                    {"name": "date", "type": "DATE", "description": "Sale date"},
                    {"name": "customer_id", "type": "INTEGER", "description": "FK to customers"},
                    {"name": "product_id", "type": "INTEGER", "description": "FK to products"},
                    {"name": "quantity", "type": "INTEGER", "description": "Units sold"},
                    {"name": "amount", "type": "DECIMAL", "description": "Total sale amount in USD"},
                ]
            },
            "customers": {
                "table": "customers",
                "columns": [
                    {"name": "id", "type": "INTEGER"},
                    {"name": "name", "type": "VARCHAR"},
                    {"name": "email", "type": "VARCHAR"},
                    {"name": "joined_date", "type": "DATE"},
                ]
            },
            "products": {
                "table": "products",
                "columns": [
                    {"name": "id", "type": "INTEGER"},
                    {"name": "name", "type": "VARCHAR"},
                    {"name": "category", "type": "VARCHAR"},
                    {"name": "price", "type": "DECIMAL"},
                ]
            }
        }
        return json.dumps(schemas.get(table_name, {"error": f"Table '{table_name}' not found. Available: {list(schemas.keys())}"}))
    
    def save_report(self, title, content):
        report_id = f"RPT-{len(self.reports) + 1:03d}"
        self.reports.append({"id": report_id, "title": title, "content": content})
        return json.dumps({"report_id": report_id, "status": "saved"})
    
    def process_tool(self, name, input_data):
        handlers = {
            "query_database": self.query_database,
            "calculate": self.calculate,
            "create_chart": self.create_chart,
            "get_table_schema": self.get_table_schema,
            "save_report": self.save_report,
        }
        handler = handlers.get(name)
        if not handler:
            return f"Unknown tool: {name}"
        try:
            return handler(**input_data)
        except Exception as e:
            return f"Error: {e}"
    
    def run(self, user_message, max_turns=15):
        messages = [{"role": "user", "content": user_message}]
        
        for _ in range(max_turns):
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=4096,
                tools=self.tools,
                system="You are a data analyst assistant. Use the tools to explore data, run queries, and provide insights.",
                messages=messages
            )
            
            if response.stop_reason == "end_turn":
                return next((b.text for b in response.content if b.type == "text"), "")
            
            messages.append({"role": "assistant", "content": response.content})
            
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = self.process_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            
            messages.append({"role": "user", "content": tool_results})
        
        return "Max turns reached"


# Test
agent = DataAnalysisAgent()

print("=== Test 1: Schema exploration ===")
print(agent.run("What tables are in the database? Show me the schema for the sales table."))

print("\n=== Test 2: Analysis + Chart ===")
agent2 = DataAnalysisAgent()
print(agent2.run("How many sales were made in Q4 2024? Create a chart of monthly sales."))

---

## Challenge 3: DevOps Incident Agent (Target: 35 min)

### Scenario
Build an agent that helps investigate and respond to production incidents.

### Requirements
1. Define **6 tools**:
   - `check_service_health(service_name)` — Is the service up/down?
   - `get_error_logs(service_name, minutes)` — Recent error logs
   - `get_metrics(service_name, metric_name)` — CPU, memory, latency metrics
   - `restart_service(service_name)` — Restart a service (requires confirmation)
   - `scale_service(service_name, replicas)` — Scale up/down
   - `create_incident(title, severity, description)` — Create incident ticket

2. **Safety**: `restart_service` should require a `confirmed: true` parameter

3. **State**: Track which services have been restarted, incidents created

4. Test: "The API is slow. Investigate and fix it."

This is a harder challenge because Claude needs to:
- Check health → check logs → check metrics → decide action → take action → create incident

In [ ]:
# ============================================
# CHALLENGE 3: YOUR CODE HERE
# ============================================



In [ ]:
# ============================================
# CHALLENGE 3: SOLUTION
# ============================================

class IncidentAgent:
    def __init__(self):
        self.restarted_services = []
        self.incidents = []
        
        self.tools = [
            {
                "name": "check_service_health",
                "description": "Check if a service is healthy, degraded, or down. Returns status and uptime.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "service_name": {"type": "string", "description": "Name of the service, e.g. 'api-gateway', 'user-service'"}
                    },
                    "required": ["service_name"]
                }
            },
            {
                "name": "get_error_logs",
                "description": "Get recent error logs for a service. Returns log entries from the last N minutes.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "service_name": {"type": "string"},
                        "minutes": {"type": "integer", "description": "How many minutes back to look. Default 30."}
                    },
                    "required": ["service_name"]
                }
            },
            {
                "name": "get_metrics",
                "description": "Get performance metrics for a service. Metric options: cpu, memory, latency, error_rate.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "service_name": {"type": "string"},
                        "metric_name": {"type": "string", "enum": ["cpu", "memory", "latency", "error_rate"]}
                    },
                    "required": ["service_name", "metric_name"]
                }
            },
            {
                "name": "restart_service",
                "description": (
                    "Restart a service. This is a disruptive action. "
                    "The 'confirmed' parameter must be set to true to proceed. "
                    "Only use this after investigating the issue and determining a restart is needed."
                ),
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "service_name": {"type": "string"},
                        "confirmed": {"type": "boolean", "description": "Must be true to confirm restart"}
                    },
                    "required": ["service_name", "confirmed"]
                }
            },
            {
                "name": "scale_service",
                "description": "Scale a service to a specific number of replicas. Use to handle increased load.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "service_name": {"type": "string"},
                        "replicas": {"type": "integer", "description": "Target number of replicas"}
                    },
                    "required": ["service_name", "replicas"]
                }
            },
            {
                "name": "create_incident",
                "description": "Create an incident ticket to track the issue. Use after investigating.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "title": {"type": "string", "description": "Incident title"},
                        "severity": {"type": "string", "enum": ["low", "medium", "high", "critical"]},
                        "description": {"type": "string", "description": "Details of the incident"}
                    },
                    "required": ["title", "severity", "description"]
                }
            }
        ]
    
    def check_service_health(self, service_name):
        statuses = {
            "api-gateway": {"status": "degraded", "uptime": "99.2%", "response_time_ms": 2500},
            "user-service": {"status": "healthy", "uptime": "99.99%", "response_time_ms": 45},
            "database": {"status": "healthy", "uptime": "99.95%", "connections": 85},
            "cache": {"status": "healthy", "uptime": "99.99%", "hit_rate": "92%"},
        }
        return json.dumps(statuses.get(service_name, {"status": "unknown", "error": "Service not found"}))
    
    def get_error_logs(self, service_name, minutes=30):
        logs = {
            "api-gateway": [
                {"time": "14:23:01", "level": "ERROR", "message": "Connection pool exhausted"},
                {"time": "14:23:15", "level": "ERROR", "message": "Timeout waiting for upstream: user-service"},
                {"time": "14:24:02", "level": "WARN", "message": "Memory usage above 85%"},
                {"time": "14:25:00", "level": "ERROR", "message": "Connection pool exhausted"},
            ]
        }
        return json.dumps({"service": service_name, "period_minutes": minutes, 
                          "logs": logs.get(service_name, []), "count": len(logs.get(service_name, []))})
    
    def get_metrics(self, service_name, metric_name):
        metrics = {
            "api-gateway": {"cpu": "78%", "memory": "87%", "latency": "2500ms", "error_rate": "12%"},
            "user-service": {"cpu": "25%", "memory": "40%", "latency": "45ms", "error_rate": "0.1%"},
        }
        service_metrics = metrics.get(service_name, {})
        return json.dumps({"service": service_name, "metric": metric_name, 
                          "value": service_metrics.get(metric_name, "N/A")})
    
    def restart_service(self, service_name, confirmed=False):
        if not confirmed:
            return json.dumps({"error": "Restart requires confirmed=true. This is a disruptive action."})
        self.restarted_services.append(service_name)
        return json.dumps({"service": service_name, "status": "restarted", "message": "Service restarted successfully"})
    
    def scale_service(self, service_name, replicas):
        return json.dumps({"service": service_name, "replicas": replicas, "status": "scaled"})
    
    def create_incident(self, title, severity, description):
        inc_id = f"INC-{len(self.incidents) + 1:03d}"
        self.incidents.append({"id": inc_id, "title": title, "severity": severity})
        return json.dumps({"incident_id": inc_id, "title": title, "severity": severity, "status": "created"})
    
    def process_tool(self, name, input_data):
        handlers = {
            "check_service_health": self.check_service_health,
            "get_error_logs": self.get_error_logs,
            "get_metrics": self.get_metrics,
            "restart_service": self.restart_service,
            "scale_service": self.scale_service,
            "create_incident": self.create_incident,
        }
        handler = handlers.get(name)
        if not handler:
            return f"Unknown tool: {name}"
        try:
            return handler(**input_data)
        except Exception as e:
            return f"Error: {e}"
    
    def run(self, user_message, max_turns=15):
        messages = [{"role": "user", "content": user_message}]
        for _ in range(max_turns):
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=4096,
                tools=self.tools,
                system=(
                    "You are a DevOps incident response agent. When investigating issues, "
                    "check health first, then logs, then metrics. Take corrective action "
                    "if needed, and always create an incident ticket to document the issue."
                ),
                messages=messages
            )
            if response.stop_reason == "end_turn":
                return next((b.text for b in response.content if b.type == "text"), "")
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = self.process_tool(block.name, block.input)
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
            messages.append({"role": "user", "content": tool_results})
        return "Max turns reached"


# Test
agent = IncidentAgent()
print(agent.run("The API is slow. Investigate and fix it."))
print(f"\nRestarted: {agent.restarted_services}")
print(f"Incidents: {agent.incidents}")

---

## Post-Coding Discussion Prep

After the coding portion, you'll discuss your design. Here are likely questions and strong answers:

### Q: "Walk me through your agentic loop."
**A**: The loop sends messages to Claude, checks `stop_reason`. If it's `tool_use`, I extract all tool_use blocks, execute each tool via my dispatcher, collect results into tool_result blocks in a single user message, and continue the loop. If `stop_reason` is `end_turn`, Claude is done and I return the text response. I have a `max_turns` safety limit.

### Q: "How do you handle errors?"
**A**: Tool execution errors are caught by try/except in the dispatcher and returned to Claude as tool results (optionally with `is_error: true`). Claude can then retry with different parameters or inform the user about the error. I don't crash the loop on tool errors.

### Q: "Why did you structure your tools this way?"
**A**: I made tools specific and well-described so Claude knows when to use each one. Each tool has a single responsibility. The descriptions tell Claude not just what the tool does, but when to use it and what it returns.

### Q: "What would you change for production?"
**A**: Rate limiting, authentication, proper logging/observability, input validation/sanitization, timeouts on tool execution, and potentially streaming for long responses. I'd also add proper error types and retry logic with exponential backoff.

### Q: "How would you test this?"
**A**: Unit tests for individual tool handlers. Integration tests with mocked API responses. For the agentic loop, I'd test with deterministic tool inputs to verify the message history is constructed correctly.

### Q: "What are the limitations of this approach?"
**A**: Context window limits (long conversations accumulate tokens), cost per API call, latency for multi-step tasks, potential for hallucinated tool parameters, and no way to "undo" tool actions. Also, the model might get stuck in loops calling the same tool repeatedly.

---

## Final Checklist — Before the Interview

Can you do these from memory? If not, practice the relevant notebook.

- [ ] Set up `anthropic.Anthropic()` client
- [ ] Write a tool definition with name, description, input_schema
- [ ] Use JSON Schema: string, number, integer, boolean, enum, array, required
- [ ] Send a message with tools to the API
- [ ] Check `response.stop_reason == "tool_use"`
- [ ] Extract `tool_use` blocks from `response.content`
- [ ] Build a `tool_result` message with matching `tool_use_id`
- [ ] Write the agentic while/for loop
- [ ] Handle parallel tool calls (multiple tool_use blocks → single user message with all results)
- [ ] Write a dispatcher (dict mapping tool names to functions)
- [ ] Handle errors with try/except and `is_error: true`
- [ ] Use `**input_data` to unpack tool input as keyword arguments
- [ ] Explain your design decisions clearly

**Good luck!**